# Bài 34 — chuỗi luyện Easy · LeetCode #171 — Excel Sheet Column Number

**Giáo trình Jupyter Notebook tiếng Việt cho người mới bắt đầu · Python 3**  
**Link đề:** https://leetcode.com/problems/excel-sheet-column-number/ · **Mức độ:** Easy · **Chủ điểm:** String · Math · đổi cơ số 26 kiểu 1-based

**Kết quả bạn đạt được sau bài học**
1. Phát biểu được chính xác input, output, và những điều kiện đề bảo đảm; phân biệt dữ liệu thật với ví dụ mô phỏng.
2. Tự diễn giải công cụ Python liên quan bằng chương trình mini; không ghi nhớ cú pháp một cách máy móc.
3. Tự nghĩ ra một cách giải dễ hiểu, thấy rõ điểm hạn chế, rồi suy ra cách chính tốt hơn.
4. Lập bảng và chạy chương trình `trace_*` để thấy trạng thái thay đổi từng bước, kể cả nhánh đặc biệt.
5. Giải thích được tính đúng bằng **bất biến (invariant)**, tự suy luận Time/Space Complexity.
6. Viết lại `class Solution` để nộp LeetCode và tự giải 6 bài tập, có gợi ý ba mức và đáp án cuối.

> **Cách học:** Đọc một markdown → tự dự đoán kết quả code → chạy cell → so sánh → tự giải thích vì sao. Các cell TODO chỉ định nghĩa hàm, không được chạy tự động.


## 1. Đề bài, ví dụ và cách hiểu đúng số cột Excel

Cho chuỗi `columnTitle` gồm các chữ in hoa A..Z, biểu diễn **một cột Excel**, hãy trả về **số thứ tự cột**. Mỗi ký tự tương ứng 1..26: `A=1`, ..., `Z=26`; `AA=27`, `AB=28`, `ZY=701`. Đề giới hạn độ dài `1..7` ký tự và giá trị không quá tên cột `FXSHRXW` = 2,147,483,647.

Các ví dụ chính thức: `"A" → 1`, `"AB" → 28`, `"ZY" → 701`. Ở bài #168, ta **chia lấy dư** để đổi số thành tên; bài này làm hướng ngược lại: **nhân 26 rồi cộng chữ mới**. Ta sẽ học công thức bằng những số cụ thể trước khi viết code.

Khác hệ số thông thường: Excel không có ký hiệu 0. Vì vậy `AA = 1×26 + 1`, không phải 0×26 +0. Đừng lấy `ord('A')` (=65) để làm giá trị cột hoặc lấy trực tiếp index 0-based của chữ A mà quên cộng 1.


## 2. Thiết lập môi trường để chạy notebook

Notebook dùng **Python 3** chuẩn, không cần cài thêm package. Hãy chạy **từ trên xuống dưới** để các tên hàm đã được định nghĩa trước khi gọi. Không copy `print(...)` vào LeetCode; các dòng in ở đây chỉ phục vụ việc học.

**Phân biệt ba lớp:** (1) demo nhỏ để hiểu công cụ, (2) hàm minh họa giải bài trong notebook, (3) `class Solution` đúng chữ ký để nộp trên trang đề. Với bài #171, ta sẽ học kỹ thuật **String · Math · đổi cơ số 26 kiểu 1-based** theo đúng thứ tự này.

Từng code cell sau đây cần cho ra output ngắn và có thể kiểm tra bằng mắt. Trong cell kiểm thử, `assert` không có output nghĩa là điều kiện đã đúng; nếu sai, Python báo `AssertionError`.


In [1]:
from __future__ import annotations
import random
ALPHABET='ABCDEFGHIJKLMNOPQRSTUVWXYZ'
print('A =',ALPHABET.index('A')+1,'Z =',ALPHABET.index('Z')+1)
print('AB = 1*26 + 2 =',1*26+2)


A = 1 Z = 26
AB = 1*26 + 2 = 28


### 2.1. `str`, indexing và duyệt từng ký tự

`columnTitle` là string, ví dụ `"AB"`. Vòng `for ch in columnTitle` lần lượt đọc A rồi B, không cần gọi `split()`. `s[0]` là ký tự đầu, `s[-1]` là cuối. Chuỗi Python bất biến: ta chỉ đọc từng chữ, không cập nhật `s[i]`.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [2]:
s='ZY'
print('len=',len(s),'đầu=',s[0],'cuối=',s[-1])
for i,ch in enumerate(s):print('i=',i,'ch=',ch)


len= 2 đầu= Z cuối= Y
i= 0 ch= Z
i= 1 ch= Y


### 2.2. `ord()` đọc mã, không phải thứ tự cột

`ord("A")=65`, `ord("B")=66`; **giá trị cột** của một chữ là `ord(ch)-ord("A")+1`, nghĩa là A→1, B→2, Z→26. Chúng ta dùng `ord` vì Unicode của các chữ in hoa tiếng Anh liên tiếp; đó là thuộc tính ký tự, không phải thuật toán magic.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [3]:
for ch in ['A','B','M','Z']:
    code=ord(ch)
    digit=code-ord('A')+1
    print(ch,'Unicode=',code,'Excel digit=',digit)


A Unicode= 65 Excel digit= 1
B Unicode= 66 Excel digit= 2
M Unicode= 77 Excel digit= 13
Z Unicode= 90 Excel digit= 26


### 2.3. Cơ số 26: mỗi chữ phía trước mạnh hơn 26 lần

Chẳng hạn `AB = 1×26 + 2 =28`; `ZY = 26×26 + 25 = 701`. Nếu thêm một chữ mới vào sau một tiền tố đang có giá trị x, toàn bộ tiền tố dịch trái một bậc → nhân 26, rồi thêm digit mới. Đây là lý do xuất hiện `value=value*26+digit`.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [4]:
print('AB:',1*26+2)
print('ZY:',26*26+25)
print('AAA:',1*26**2+1*26+1)


AB: 28
ZY: 701
AAA: 703


### 2.4. Bất biến của tổng tích lũy

Sau khi đọc k chữ đầu, biến `value` biểu diễn **giá trị của đúng k chữ ấy**. Bắt đầu với 0 vì chưa có chữ; khi đọc d (1..26), cập nhật `value*26 + d`. Không cần biết độ dài chuỗi trước khi bắt đầu hoặc phân tích số lượng ký tự.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [5]:
value=0
for ch in 'AB':
    digit=ord(ch)-ord('A')+1
    before=value;value=value*26+digit
    print(before,'*26 +',digit,'=',value)


0 *26 + 1 = 1
1 *26 + 2 = 28


### 2.5. `reversed` và mũ `**` trong cách giải baseline

Một cách tính tay là tính từng chữ kèm trọng số `26**power`. Với `"ZY"`, Z ở hàng 26¹, Y ở hàng 26⁰. Dùng `reversed(s)` để đi từ phải qua trái, `enumerate` để lấy `power` tương ứng. `**` là lũy thừa, không phải XOR; dấu `^` trong Python là XOR!

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [6]:
s='ZY';ans=0
for power,ch in enumerate(reversed(s)):
    digit=ord(ch)-64
    part=digit*(26**power)
    print(ch,'trọng số 26^',power,'đóng góp',part)
    ans+=part
print('total=',ans)


Y trọng số 26^ 0 đóng góp 25
Z trọng số 26^ 1 đóng góp 676
total= 701


### 2.6. `+=`, dấu `==` và kiểm tra dữ liệu

`total += digit` là cập nhật tổng, còn `total == expected` tạo boolean. Những lỗi như cộng nhầm phần tử, quên nhân 26, hoặc nhân sau khi cộng sẽ tạo các số rất khác ngay ở `AB`. Chỉ dùng input A..Z theo constraints, không tự thêm xử lý lowercase nếu đề không yêu cầu.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [7]:
total=0
for digit in [1,2]:
    total=total*26+digit
print('AB:',total,'đúng?',total==28)


AB: 28 đúng? True


### 2.7. `int` của Python và phân tích theo số chữ

Python `int` có thể chứa số lớn, không cần `int32` thủ công. Đề giới hạn tới 2³¹-1, nhưng thuật toán không phụ thuộc một số hằng. Nếu chuỗi có k ký tự, vòng lặp chạy k lần; phân tích theo **k** thay vì giá trị cột vì đọc một chữ mỗi lượt.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [8]:
for s in ['A','Z','AA','ZZ','AAA','FXSHRXW']:
    print(s,'độ dài input=',len(s),'chữ')


A độ dài input= 1 chữ
Z độ dài input= 1 chữ
AA độ dài input= 2 chữ
ZZ độ dài input= 2 chữ
AAA độ dài input= 3 chữ
FXSHRXW độ dài input= 7 chữ


## 3. Nền tảng tư duy — từ ví dụ nhỏ đến bài toán

Bảy thí nghiệm ở trên không phải những mẹo rời rạc: chúng chính là các khối xây dựng cho bài #171. Hãy kiểm tra lần lượt: **dữ liệu đang nằm ở đâu, biến nào lưu thông tin đã biết, bước nào khiến ta tiến tới đáp án?**

Khi học thuật toán, đừng chỉ hỏi “code này chạy gì?”; hãy hỏi **tại sao không cần nhớ phần thông tin đã bỏ đi** và vì sao ta không bỏ sót lời giải. Phần sau sẽ chạy một số ví dụ rất nhỏ trước khi ghép thành hàm hoàn chỉnh.


In [9]:
def digit_171(ch):
    return ord(ch)-ord('A')+1
for s in ['A','AB','ZY']:
    print('chuỗi',s,'các chữ số Excel:',[digit_171(ch) for ch in s])


chuỗi A các chữ số Excel: [1]
chuỗi AB các chữ số Excel: [1, 2]
chuỗi ZY các chữ số Excel: [26, 25]


### Câu hỏi gợi mở trước khi tối ưu

- Có thể tính "ZY" nếu không biết s có bao nhiêu ký tự trước không?
- Vì sao thêm chữ B sau A làm số tăng từ 1 lên 28, không phải thành 3?
- Với `"FXSHRXW"`, cần tạo list các lũy thừa 26 trước hay chỉ một biến tổng là đủ?

Chưa cần thuộc tên kỹ thuật **String · Math · đổi cơ số 26 kiểu 1-based**. Trước tiên hãy tưởng tượng bạn chỉ có bút và giấy: bạn đọc input theo thứ tự nào, giữ lại những gì, điều gì buộc bạn dừng? Câu trả lời sẽ trở thành các biến và điều kiện `if`/`while`.


### Quy ước chạy cell và cách tự kiểm tra

- Đọc thứ tự từ trên xuống: các hàm helper trong phần này chỉ dùng để dựng dữ liệu hoặc hiển thị; đáp án submit sẽ không phụ thuộc chúng.
- `print` là quan sát, `return` là kết quả hàm. Dùng `assert actual == expected` để kiểm chứng, không thay thế một phép kiểm bằng việc nhìn thấy chữ PASS.
- Khi thấy nhiều thuật toán, chúng có **tên hàm khác nhau** (`baseline_*`, `solve_*`, `trace_*`) để tránh ghi đè logic. Không thay đổi input trừ khi đề cho phép.
- Đừng chạy hàm TODO trước khi tự hoàn thiện: hàm được để `pass`/`NotImplementedError` có chủ đích.

**Nguyên tắc quan trọng:** ví dụ và test là bằng chứng thử nghiệm; lập luận về bất biến mới giúp giải thích vì sao thuật toán đúng cho mọi input hợp lệ.


### 3.1. Gắn giá trị 1..26 cho từng ký tự

Ta ánh xạ mỗi ch→digit độc lập bằng `ord(ch)-ord('A')+1`. Điều này biến bài chuỗi thành bài cộng số theo vị trí; không cần mảng dict 26 phần tử (dù dùng dict cũng hợp lệ). Thử A, Z và các chữ giữa; cùng một công thức áp dụng cả 26 chữ.

Chạy đoạn code dưới rồi thử tự diễn đạt **trạng thái trước – thao tác – trạng thái sau**. Nếu chỉ nhớ kết quả in ra mà không giải thích được sự chuyển đổi thì nên đọc lại ví dụ.


In [10]:
for ch in 'ABZY':
    print(ch,'->',digit_171(ch))
assert digit_171('A')==1 and digit_171('Z')==26


A -> 1
B -> 2
Z -> 26
Y -> 25


### 3.2. Tính từng tiền tố của "FXSHRXW"

Dùng `value=0`, đọc F→X→S→H→R→X→W. Sau mỗi chữ, `value` chính là số cột của **tiền tố đã đọc**, không phải giá trị cuối của cả chuỗi. Vì thế `value` giữ lại toàn bộ thông tin cần cho chữ kế tiếp mà không cần lưu từng bước trước đó.

Đây là bước nối giữa ví dụ nhỏ và code thuật toán: những biến xuất hiện trong cell tiếp theo sẽ tái xuất hiện trong `solve_*` hoặc `trace_*` ở dưới.


In [11]:
v=0
for ch in 'FXSHRXW':
    old=v
    v=26*v+digit_171(ch)
    print(f'{old} × 26 + {digit_171(ch)} = {v}')
assert v==2147483647


0 × 26 + 6 = 6
6 × 26 + 24 = 180
180 × 26 + 19 = 4699
4699 × 26 + 8 = 122182
122182 × 26 + 18 = 3176750
3176750 × 26 + 24 = 82595524
82595524 × 26 + 23 = 2147483647


## 4. Cách giải trực giác / Baseline

Baseline dễ liên hệ với toán phổ thông: tính tổng theo các vị trí có trọng số 26^k. Với `s="ZY"`, ta làm `26×26 + 25×1`. Duyệt `reversed(s)`, dùng `enumerate` để lấy mũ, cộng dồn. Cách này đúng nhưng phải quan tâm mũ và tạo phép lũy thừa nhiều lần, trong khi đọc trái→phải có thể đơn giản hơn.

Ta cố ý viết phiên bản **dễ nghĩ nhất trước**, dù đôi khi nó dùng thêm bộ nhớ hoặc thao tác dư thừa. Lợi ích của baseline: nhìn được định nghĩa bài toán thành code, và có một “đáp án chuẩn” để đối chiếu khi tối ưu.


In [12]:
def baseline_171(columnTitle):
    total=0
    for power,ch in enumerate(reversed(columnTitle)):
        digit=ord(ch)-ord('A')+1
        total+=digit*(26**power)
    return total


In [13]:
for s in ['A','Z','AA','AB','ZY','ZZ','AAA','FXSHRXW']:
    print(s,'=>',baseline_171(s))


A => 1
Z => 26
AA => 27
AB => 28
ZY => 701
ZZ => 702
AAA => 703
FXSHRXW => 2147483647


### 4.1. Đọc baseline và nhận ra chi phí

Baseline thực hiện k lần với k=len(s), nhưng tính lũy thừa và `reversed` có thể thêm thao tác; trong mô hình số nguyên kích thước cố định, thời gian O(k). Python dùng `reversed(s)` là iterator, không tạo bản sao chuỗi; bộ nhớ phụ O(1) trong cách này (không lưu các chữ đảo). Khi phân tích số nguyên cực lớn, chi phí phép nhân/lũy thừa phụ thuộc độ dài bit, nhưng bài này giới hạn số ≤2^31-1 nên mô hình thao tác số hằng là phù hợp.

Không nên khẳng định nhanh/chậm chỉ dựa trên việc chạy vài ví dụ. Hãy **đếm bước**, chỉ ra `n` đang đo cái gì và xét input xấu nhất. Sau này khi thử tối ưu, kết quả phải vẫn khớp baseline với cùng một input.


In [14]:
for s in ['AB','ZY','AAA']:
    terms=[]
    for power,ch in enumerate(reversed(s)):
        terms.append(f'{digit_171(ch)}*26^{power}')
    print(s,':',' + '.join(reversed(terms)),'=',baseline_171(s))


AB : 1*26^1 + 2*26^0 = 28
ZY : 26*26^1 + 25*26^0 = 701
AAA : 1*26^2 + 1*26^1 + 1*26^0 = 703


### 4.2. Đường vòng / phản ví dụ giúp thấy lỗi tư duy

Một cách sai: cộng giá trị từng chữ, ví dụ `A(1)+B(2)=3`, không phải 28. Một cách khác là dùng `ord(ch)-ord('A')` (A=0) khiến `A→0`. Cần cả **nhân 26** và **dịch 1-based**, không bỏ một trong hai.

Hãy tìm một đầu vào **cực nhỏ** làm giả thuyết sai; dạng test này hiệu quả hơn việc chỉ chạy ngẫu nhiên dữ liệu dài.


In [15]:
print('A+B sai nếu chỉ cộng:',1+2,'; đúng:',1*26+2)
wrong_a=ord('A')-ord('A')
print('A sai nếu không +1:',wrong_a,'; đúng:',digit_171('A'))
assert baseline_171('AB')==28


A+B sai nếu chỉ cộng: 3 ; đúng: 28
A sai nếu không +1: 0 ; đúng: 1


## 5. Vì sao cần bước cải tiến?

Baseline đã đúng nhưng việc tính mũ làm khó trace và dễ nhầm chỉ số. Khi ta đọc từ trái sang phải, toàn bộ tiền tố hiện tại có giá trị x; mỗi chữ mới d chỉ cần `x*26 + d`. Không cần tính `26**power` hay biết vị trí cách cuối chuỗi bao xa. Đó chính là cách đọc số ở mọi hệ đếm vị trí.

Ta không muốn “tối ưu bằng phép màu”. Mục tiêu là **chứng minh có thể bỏ một phần công việc** mà vẫn giữ thông tin đủ để quyết định đúng. Sau khi tìm thấy đại lượng/cấu trúc bắt buộc phải duy trì, ta mới viết thuật toán chính.


### Bước đột phá — Giá trị tiền tố mới = giá trị cũ ×26 + chữ số 1..26

Với `AB`: ban đầu x=0; đọc A (digit=1), x=0×26+1=1; đọc B (digit=2), x=1×26+2=28. Với `ZY`: đọc Z x=26; đọc Y x=26×26+25=701. Hãy thử `AAA` để thấy phép nhân lặp lại lần thứ ba.

**Tự kiểm tra:** Nếu làm thêm đúng một bước nữa, cái gì chắc chắn vẫn đúng? Nếu thuật toán dừng ngay bây giờ, vì sao kết quả nó trả là hợp lệ? Hai câu hỏi này chính là hạt nhân của bất biến và điều kiện dừng.


## 6. Thuật toán chính — mổ xẻ từng bước rồi ghép lại

Hàm chính rất ngắn: `value=0`, vòng `for ch in columnTitle` tính digit, cập nhật `value=value*26+digit`, cuối cùng `return value`. Các bước ngắn vì ta đã hiểu lý do. Trong cell sau, từng phép tính được in trước khi viết hàm không debug.

Trước tiên xem một phép biến đổi ngắn trong cell nhỏ, sau đó đọc hàm `solve_*` không có `print`. **Không nhảy thẳng đến class Solution**: phần luyện tay và trace phía dưới sẽ làm rõ từng điều kiện.


### 6.1. Đọc từng biến, không chỉ đọc tên hàm

- `total=0`: chưa đọc ký tự nào.
- `for ch in columnTitle`: duyệt trái sang phải đúng thứ tự chữ trong tiêu đề.
- `digit=ord(ch)-ord('A')+1`: ánh xạ A..Z thành 1..26.
- `total=total*26+digit`: chữ mới ở hàng đơn vị, tiền tố cũ dịch trái 1 bậc (nhân 26).
- `return total`: kết quả là số nguyên theo đúng chữ ký của LeetCode.

Đặc biệt để ý **thứ tự cập nhật**: đổi thứ tự hai dòng tưởng nhỏ có thể thay đổi ý nghĩa của bất biến. Nếu nhầm, hãy viết trạng thái trước và sau bằng một ví dụ ba phần tử / ba bước.


In [16]:
value=1  # A
next_digit=2  # B
print('AB =',value,'*26 +',next_digit,'=',value*26+next_digit)
assert value*26+next_digit==28


AB = 1 *26 + 2 = 28


In [17]:
def solve_171(columnTitle: str) -> int:
    total=0
    for ch in columnTitle:
        digit=ord(ch)-ord('A')+1
        total=total*26+digit
    return total


### 6.2. Tại sao các nhánh `if`/`while` được viết đúng thứ tự?

Thứ tự “nhân trước, cộng sau” đảm bảo chữ mới chiếm hàng đơn vị; nếu cộng trước rồi nhân sẽ làm `AB` thành `(1+2)*26=78`, sai. Vòng `for` không bỏ qua ký tự đầu/cuối. Dùng `str` uppercase đúng constraints, vì `ord('a')` sẽ mang giá trị ngoài 1..26.

Trong một vòng lặp hãy đọc bằng câu tiếng Việt: “**Trước bước này ta đã biết ...; ta kiểm tra ...; nếu ... thì ...; sau bước này ...**”. Có thể thêm `print` để kiểm tra suy luận, nhưng hàm nộp bài phải trả **đúng kiểu** kết quả.


In [18]:
for s in ['A','B','Z','AA','AB','ZY','AAA','FXSHRXW']:
    assert solve_171(s)==baseline_171(s)
    print(s,'=',solve_171(s))


A = 1
B = 2
Z = 26
AA = 27
AB = 28
ZY = 701
AAA = 703
FXSHRXW = 2147483647


### 6.3. Kiểm tra mấu chốt bằng vài bước tay

Sau khi đọc k chữ đầu, `total` là giá trị cột được biểu diễn bởi **tiền tố dài k**. Ban đầu tiền tố rỗng có giá trị 0. Khi ghép ký tự mới với digit d, theo quy tắc hệ 26, giá trị mới bằng `26*total+d`. Sau k=len(s), tiền tố là toàn bộ chuỗi nên `total` là đáp án.

Một bất biến đúng phải có ba điều: **đúng lúc bắt đầu; vẫn đúng sau một lần cập nhật; khi dừng thì suy ra đáp án**. Dù chưa biết từ “invariant”, bạn có thể phát biểu tự nhiên như câu trên.


In [19]:
total=0
for k,ch in enumerate('ZY',start=1):
    before=total
    total=total*26+digit_171(ch)
    print('đã đọc',k,'chữ, trước',before,'sau',total)
assert total==701


đã đọc 1 chữ, trước 0 sau 26
đã đọc 2 chữ, trước 26 sau 701


## 7. TRACE từng bước — nhìn thấy trạng thái biến

Ta sẽ dùng chương trình `trace_*` khác với lời giải chính: `solve_*` phục vụ submit và test; `trace_*` chỉ phục vụ việc **quan sát**. Trace phải cho thấy dữ liệu đầu vào, biến *trước khi cập nhật*, điều kiện/nhánh đang xét, biến *sau khi cập nhật*, và điều kiện dừng.

Một trace tốt phải phân biệt cột `digit` và `total`: `digit` chỉ là 1..26 của chữ hiện tại; `total` là **toàn bộ tiền tố**. Bảng với `ZY` có hai dòng: `0*26+26=26`, rồi `26*26+25=701`.


In [20]:
def trace_171(s):
    total=0
    print('k | ký_tự | digit | total_trước | phép_tính | total_sau')
    for k,ch in enumerate(s,1):
        before=total
        digit=ord(ch)-ord('A')+1
        total=26*total+digit
        print(f'{k:1d} | {ch:^6s} | {digit:5d} | {before:11d} | {before}*26+{digit} | {total}')
    print('DỪNG ->',total)
    return total
assert trace_171('ZY')==701


k | ký_tự | digit | total_trước | phép_tính | total_sau
1 |   Z    |    26 |           0 | 0*26+26 | 26
2 |   Y    |    25 |          26 | 26*26+25 | 701
DỪNG -> 701


### 7.1. Trace dạng bảng — diễn giải cột và thứ tự cập nhật

Từng hàng mô tả một ký tự **trước khi** cập nhật. Hàng Z: trước=0, digit=26, sau=26. Hàng Y: trước=26, digit=25, sau=701. Khi sang hàng kế tiếp, `total_trước` phải khớp `total_sau` của hàng trước. Đây là cách soi lỗi rất trực tiếp.

Khi đọc bảng, đừng coi một hàng là một đáp án; một hàng biểu diễn **một phép chuyển trạng thái**. Hãy tự che cột cuối, dự đoán hành động, rồi đối chiếu với print phía dưới.


In [21]:
assert trace_171('AB')==28
assert trace_171('AAA')==703


k | ký_tự | digit | total_trước | phép_tính | total_sau
1 |   A    |     1 |           0 | 0*26+1 | 1
2 |   B    |     2 |           1 | 1*26+2 | 28
DỪNG -> 28
k | ký_tự | digit | total_trước | phép_tính | total_sau
1 |   A    |     1 |           0 | 0*26+1 | 1
2 |   A    |     1 |           1 | 1*26+1 | 27
3 |   A    |     1 |          27 | 27*26+1 | 703
DỪNG -> 703


### 7.2. Một nhánh đặc biệt mà cách học thuộc code dễ bỏ lỡ

Các ký tự Z có digit=26, **không phải 0**. Vì vậy `Z` trả 26, `AZ` trả 52, `ZZ` trả 702. Nếu lấy `ord(ch)-ord('A')`, mọi tên có A sẽ bị giảm sai, và Z bị coi là 25.

Câu hỏi sau khi chạy: **Nếu bỏ điều kiện xử lý nhánh này, input nhỏ nhất nào trả sai?**


In [22]:
for s,expected in [('Z',26),('AZ',52),('ZZ',702),('AAA',703)]:
    print('TRACE',s)
    assert trace_171(s)==expected


TRACE Z
k | ký_tự | digit | total_trước | phép_tính | total_sau
1 |   Z    |    26 |           0 | 0*26+26 | 26
DỪNG -> 26
TRACE AZ
k | ký_tự | digit | total_trước | phép_tính | total_sau
1 |   A    |     1 |           0 | 0*26+1 | 1
2 |   Z    |    26 |           1 | 1*26+26 | 52
DỪNG -> 52
TRACE ZZ
k | ký_tự | digit | total_trước | phép_tính | total_sau
1 |   Z    |    26 |           0 | 0*26+26 | 26
2 |   Z    |    26 |          26 | 26*26+26 | 702
DỪNG -> 702
TRACE AAA
k | ký_tự | digit | total_trước | phép_tính | total_sau
1 |   A    |     1 |           0 | 0*26+1 | 1
2 |   A    |     1 |           1 | 1*26+1 | 27
3 |   A    |     1 |          27 | 27*26+1 | 703
DỪNG -> 703


### 7.3. Trường hợp biên — không chỉ là dữ liệu “đẹp”

`columnTitle` không rỗng theo đề; dài tối đa 7 và chỉ gồm A..Z. Ta không tự thêm `strip()`, `.upper()` hay kiểm tra lỗi ngoài constraints, vì những thao tác đó làm mờ bản chất bài gốc. Các test lớn chú trọng tiền tố dài chứ không cần cấp list lớn.

Trường hợp biên không phải ngoại lệ vô nghĩa: nó bộc lộ một giả định ẩn trong điều kiện lặp hoặc cách tính chỉ số. Chỉ thêm xử lý khi ràng buộc thực sự cho phép trường hợp đó.


In [23]:
for s in ['A','ZZ','AAA','FXSHRXW']:
    print(s,'->',solve_171(s),'| len=',len(s))
assert solve_171('FXSHRXW')==2147483647


A -> 1 | len= 1
ZZ -> 702 | len= 2
AAA -> 703 | len= 3
FXSHRXW -> 2147483647 | len= 7


### 7.4. Đối chiếu baseline và tối ưu trên một bộ input

Hàm `baseline_171` dùng tổng trọng số 26^power, `solve_171` dùng tổng tích lũy. Chúng khác cách duyệt nhưng phải cùng kết quả với mọi string A..Z hợp lệ. Ta test những chuỗi ngắn có nhiều Z — vì digit=26 dễ bị nhầm 0.

Nếu hai lời giải cho kết quả khác nhau, chưa kết luận ngay “baseline sai”; hãy in trạng thái tại input phản ví dụ, đối chiếu lại định nghĩa, rồi sửa mệnh đề bất biến.


In [24]:
for s in ['A','B','Z','AA','AB','AZ','ZA','ZZ','AAA','ABC','ZY']:
    assert baseline_171(s)==solve_171(s)
print('Baseline theo mũ và rolling base-26: PASS')


Baseline theo mũ và rolling base-26: PASS


## 8. Vì sao thuật toán đúng? — chứng minh ngắn nhưng đủ ý

**Bất biến:** sau khi xử lý k chữ đầu, `total` biểu diễn đúng giá trị của tiền tố k ký tự theo Excel. Bước đầu k=0: tổng 0 đúng cho chuỗi rỗng giả lập. Bước chuyển: nếu tiền tố hiện tại có giá trị x, thêm chữ có digit d∈[1,26] làm các trọng số cũ tăng hệ số 26, và chữ mới ở hàng 26⁰, nên giá trị mới là `26x+d`. Sau khi xử lý toàn bộ k=len(columnTitle), biến `total` bằng giá trị cột cần trả. Vòng lặp hữu hạn, không sửa input.

**Phân biệt:** `assert` minh họa “đã chạy đúng các ca này”, không chứng minh hết vô hạn input. Chứng minh dựa vào sự thật được giữ qua mọi lần lặp và điều kiện dừng; đó là kỹ năng sẽ dùng lại ở nhiều bài LeetCode khác.


## 9. Code nộp LeetCode (Python 3)

Đây là phiên bản **không in debug**, trả về đúng dữ liệu cần thiết, giữ đúng tên phương thức `titleToNumber` và tên tham số của đề. Khi submit, copy **chỉ** `class Solution` (và import nếu có dùng); với bài dùng `ListNode`, LeetCode đã cung cấp định nghĩa kiểu node.

Sau cell này, notebook sẽ gọi chính `Solution().titleToNumber(...)` bằng `assert` để kiểm tra cách đóng gói, tránh trường hợp hàm minh họa đúng nhưng chữ ký nộp sai.


In [25]:
class Solution:
    def titleToNumber(self, columnTitle: str) -> int:
        total=0
        for ch in columnTitle:
            total=total*26 + ord(ch)-ord('A')+1
        return total


## 10. Kiểm thử: mẫu đề, biên, phản ví dụ, kiểm thử đối chiếu

Kiểm ba mẫu, ngưỡng A/Z, AA/AZ/ZA/ZZ, AAA và cận trên. Phải kiểm cả ký tự đầu/cuối: nếu quên nhân 26, `A` có thể vẫn đúng nhưng `AB` sai ngay. So `int`, không so chuỗi.

Với một ca test, viết rõ **input → expected → actual**. Ưu tiên các ca làm lộ bug thay vì lặp lại hai ví dụ thông thường. Test với input gốc theo đúng kiểu LeetCode, không chỉ kiểm tra các biến đã được biến đổi trong demo.


In [26]:
cases={'A':1,'B':2,'Z':26,'AA':27,'AB':28,'AZ':52,'BA':53,'ZY':701,'ZZ':702,'AAA':703,'FXSHRXW':2147483647}
for s,expected in cases.items():
    assert Solution().titleToNumber(s)==expected,(s,expected)
print('Kiểm thử mẫu và ngưỡng:',len(cases),'PASS')


Kiểm thử mẫu và ngưỡng: 11 PASS


### 10.1. Test hệ thống trên một miền nhỏ

Sinh mọi chuỗi dài 1..3 trên bảng chữ {A,B,Y,Z}, tổng cộng 4+16+64=84 trường hợp, so phép tính theo mũ với tổng lặp. Các chuỗi chứa Z có ích vì đây là ký hiệu có giá trị 26, trái với phần dư 0 trong hệ cơ số thường.

Với các input nhỏ, có thể duyệt toàn bộ cấu hình thỏa ràng buộc để tìm phản ví dụ. Nếu không thể duyệt tất cả do số lượng quá lớn, ta kiểm tra toàn bộ một miền con và nêu rõ phạm vi đó, **không gọi nó là chứng minh tổng quát**.


In [27]:
from itertools import product
checked=0
for k in [1,2,3]:
    for tup in product('ABYZ',repeat=k):
        s=''.join(tup)
        assert solve_171(s)==baseline_171(s)
        checked+=1
print('Vét cạn 84 chuỗi nhỏ:',checked,'PASS')


Vét cạn 84 chuỗi nhỏ: 84 PASS


### 10.2. Kiểm thử ngẫu nhiên **có seed** và so sánh với oracle

Sinh ngẫu nhiên chuỗi A..Z dài 1..7, chỉ lấy case có giá trị ≤2³¹−1 theo constraints, so oracle tính tổng lũy thừa. Seed 171 giúp lập lại cùng dữ liệu; không cần web hoặc thư viện ngoài.

`random.Random(seed)` khiến bộ test có thể chạy lại giống hệt. **Oracle** là một cách kiểm tra độc lập, thường chậm hơn nhưng dễ kiểm chứng. Không dùng hai code giống hệt nhau để đối chiếu — nếu cùng mắc lỗi, test sẽ cho cảm giác an toàn giả.


In [28]:
rng=random.Random(171);checked=0
while checked<650:
    n=rng.randint(1,7)
    s=''.join(rng.choice(ALPHABET) for _ in range(n))
    val=baseline_171(s)
    if val>2**31-1:continue
    assert solve_171(s)==val
    checked+=1
print('Random 650 chuỗi hợp lệ: PASS')


Random 650 chuỗi hợp lệ: PASS


### 10.3. Tính chất khác của đáp án và dữ liệu đầu vào

Tính chất quan trọng: nếu thêm chữ B (digit=2) vào cuối s, kết quả mới phải là `old*26+2`, miễn vẫn trong range. Với prefix không dài tới cận trên, đây là kiểm thử thuộc tính độc lập với việc gọi baseline.

Không chỉ hỏi “trả đúng số chưa?”; đôi khi đề còn yêu cầu **không sửa cấu trúc** hoặc có ràng buộc độ dài/bit. Kiểm thử thuộc tính giúp phát hiện lỗi chỉ xuất hiện ở nhóm input đặc biệt.


In [29]:
for s in ['A','AB','ZY','AAA','AAAA']:
    before=solve_171(s)
    after=solve_171(s+'B')
    assert after==before*26+2
    print(s,'→',s+'B',':',before,'→',after)
print('Tính chất nối thêm một chữ: PASS')


A → AB : 1 → 28
AB → ABB : 28 → 730
ZY → ZYB : 701 → 18228
AAA → AAAB : 703 → 18280
AAAA → AAAAB : 18279 → 475256
Tính chất nối thêm một chữ: PASS


## 11. Phân tích Time / Space Complexity

Gọi **k=len(columnTitle)**, 1≤k≤7. **Cách rolling:** một vòng `for` k lần, mỗi vòng một phép nhân và cộng, **Time O(k)**, **Extra Space O(1)**; đầu ra là một số nguyên. **Baseline mũ:** k lần, `reversed(s)` là iterator, phụ trợ O(1) theo mô hình xử lý số 32-bit; nếu tự xây bảng 26^i thì có thể dùng O(k) bộ nhớ không cần thiết. Không nên gọi thuật toán O(n) với n là số cột (giá trị số), vì số vòng phụ thuộc số chữ, không phải 2 tỷ.

Để tránh nói Big-O máy móc: chỉ rõ biến kích thước input, số vòng chạy, số bộ nhớ phụ thực sự cấp phát. Nếu đầu ra có độ dài bắt buộc, phân biệt **bộ nhớ phụ** với **bộ nhớ dành cho kết quả**; nếu 32 bit là hằng số, đừng vô tình coi đó là biến tăng vô hạn.


## 12. Sai lầm người mới hay mắc — code sai, vì sao sai, cách sửa

Lỗi điển hình: ánh xạ A=0; chỉ cộng digit không nhân 26; nhân sau khi cộng; duyệt từ phải sang trái nhưng dùng công thức tổng tích lũy trái→phải; tự xử lý lowercase trái yêu cầu đề.

Không lấy việc có kết quả đúng trên ví dụ đầu làm lý do bỏ qua các lỗi sau. Với từng lỗi, chỉ ra một input phản chứng; chạy đoạn minh họa riêng, sau đó đối chiếu với `solve_*` đã được test.


In [30]:
print('sai nếu A=0: AB ->',0*26+1)
print('sai nếu không nhân: AB ->',1+2)
print('đúng:',solve_171('AB'))


sai nếu A=0: AB -> 1
sai nếu không nhân: AB -> 3
đúng: 28


### 12.1. Lỗi thứ nhất: dùng ký tự A như số 0

`ord('A')-ord('A')=0`, nhưng Excel đánh số cột A là 1. Sai lệch sẽ lan truyền qua tất cả các chữ phía sau. Bước đúng là `+1` trong `digit = ord(ch)-ord('A')+1`.

**Cách sửa:** quay lại định nghĩa và điều kiện dừng, không chỉ thêm một `if` để vá riêng input mẫu.


In [31]:
def wrong_zero_based_171(s):
    x=0
    for ch in s:x=x*26+(ord(ch)-ord('A'))
    return x
for s in ['A','AB','ZY']:
    print(s,wrong_zero_based_171(s),'(sai) vs',solve_171(s),'(đúng)')
assert wrong_zero_based_171('A')==0


A 0 (sai) vs 1 (đúng)
AB 1 (sai) vs 28 (đúng)
ZY 674 (sai) vs 701 (đúng)


### 12.2. Lỗi thứ hai: cộng trước, nhân sau

`(x + digit)*26` coi chữ mới đứng trước hàng đơn vị, không phải nối vào sau. Với AB: bước 1 tạo 26 thay vì 1, bước 2 tạo (26+2)*26=728 thay vì 28. Cần ghi đúng `x*26+digit`.

**Cách sửa:** lập bảng một hoặc hai bước đầu để biết một phép tính dùng trạng thái *trước* hay *sau* cập nhật. Khi đã thấy lỗi, ghi lại một regression test để bug không quay lại.


In [32]:
def wrong_order_171(s):
    x=0
    for ch in s:x=(x+digit_171(ch))*26
    return x
print('AB sai:',wrong_order_171('AB'),'| đúng:',solve_171('AB'))
assert wrong_order_171('AB')!=28


AB sai: 728 | đúng: 28


## 13. Nhận diện pattern và tự suy ra kỹ thuật ở bài khác

Khi đọc một chuỗi chữ số của **hệ đếm vị trí** từ trái sang phải, thường giữ `total=total*base+digit`. Khác nhau ở cách ánh xạ ký tự thành digit: hệ cơ số 10 có 0..9; Excel dùng 1..26 nên cần +1. Bài #168 làm chiều ngược bằng chia lấy dư sau khi dịch 1.

Một kỹ thuật không gắn với duy nhất bài #171: điều cần ghi nhớ là **dấu hiệu trong đề**, **trạng thái tối thiểu cần giữ**, **quy tắc cập nhật**, **khi nào dừng**, và **mức độ phức tạp có thể đạt được**. Thử tự nói thành 4–5 câu thay vì ghi nhớ code.


## 14. Biến thể — thay đổi yêu cầu để kiểm tra hiểu thật

Dùng chính lời giải #171 để (1) đổi ngược thành tiêu đề cột số #168, và (2) hỗ trợ bảng ký hiệu cơ số bất kỳ. Khi thay bộ chữ, cần giải thích lại digit 1..k chứ không bê nguyên `ord('A')`.

Nếu hiểu thuật toán, bạn có thể thay đổi mục tiêu mà vẫn suy ra cách làm. Phần này chỉ giới thiệu biến thể **có ý nghĩa và nằm ngoài yêu cầu gốc**; code submit phía trên vẫn bám đúng đề LeetCode.


In [33]:
for s in ['A','Z','AA','ZY']:
    print('Tên cột',s,'ứng với số',solve_171(s))


Tên cột A ứng với số 1
Tên cột Z ứng với số 26
Tên cột AA ứng với số 27
Tên cột ZY ứng với số 701


### 14.1. Biến thể thứ nhất: đổi ngược một số cột thành string Excel

Đây là bài #168: mỗi vòng `divmod(n-1,26)` cho ký tự cuối ở alphabet 0-based. Ghi từng chữ vào list rồi đảo lại. Có thể kiểm tra `encode(decode(s)) == s` để phát hiện lỗi khi dùng ký hiệu Z.

Hãy xác định đầu ra mới có cần nhớ thêm trạng thái không. Nếu có, giải thích vì sao; nếu không, chỉ ra biến đang đủ dùng.


In [34]:
def encode_171(n):
    chars=[]
    while n>0:
        n,r=divmod(n-1,26)
        chars.append(chr(ord('A')+r))
    return ''.join(reversed(chars))
for s in ['A','AB','ZY','ZZ','FXSHRXW']:
    assert encode_171(solve_171(s))==s
print('Đổi ngược: PASS')


Đổi ngược: PASS


### 14.2. Biến thể thứ hai: giải mã chuỗi theo bảng ký hiệu bất kỳ

Giả sử alphabet=`"ABC"` (k=3). A=1, B=2, C=3; `BA=2*3+1=7`. Dùng `alphabet.index(ch)+1` để lấy digit, rồi `total=total*k+digit`. Bài mở rộng cần giả sử mỗi ký hiệu chỉ một ký tự và alphabet không lặp.

Khi thay đổi ràng buộc đầu vào, các chứng minh cho bài gốc có thể **không còn đúng**; cần sửa chứng minh trước khi sửa code.


In [35]:
def decode_bijective_171(s,alphabet):
    k=len(alphabet)
    value=0
    for ch in s:value=value*k+alphabet.index(ch)+1
    return value
print('ABC system: A->',decode_bijective_171('A','ABC'))
print('BA->',decode_bijective_171('BA','ABC'))
assert decode_bijective_171('BA','ABC')==7


ABC system: A-> 1
BA-> 7


## 15. Hệ thống luyện tập — 6 bài từ dễ đến nâng cao

**Bài 1 — dự đoán output:** Dự đoán `ord("A")-ord("A")+1`, `ord("Z")-ord("A")+1`, và `0*26+1`. **Đừng chạy ngay:** ghi kết quả dự đoán, rồi đối chiếu với code nền tảng.

**Bài 2 — tính tay / trace:** Tính tay `AZ`, `BA`, `AAA` bằng cột tiền tố trước → digit → tổng sau. Ghi ít nhất ba cột *trạng thái trước / thao tác / trạng thái sau*.

**Bài 3 — TODO viết lời giải chính:** hoàn thiện hàm dưới đây mà **không nhìn phần đáp án**.

**Bài 4 — biến thể:** thay đổi kiểu output hoặc điều kiện nhưng dùng lại mấu chốt thuật toán.

**Bài 5 — tự tạo kiểm thử:** nghĩ ít nhất 5 ca phân biệt được code đúng/sai, có ca biên và ca phản chứng.

**Bài 6 — mini challenge:** tự tổng quát hóa kỹ thuật, thử phân tích độ phức tạp và đối chiếu với baseline.

> Các hàm `exercise_*` bên dưới cố ý chưa có lời giải. Chúng **không được gọi tự động** khi chạy notebook. Đáp án chỉ xuất hiện ở mục 17.


### Bài 3 — TODO tự viết lời giải #171

Viết `exercise_number_171(columnTitle)` không dùng `baseline_171` hay `solve_171`. Chỉ một biến tổng, vòng for trái→phải, trả int đúng với mọi string A..Z hợp lệ.

Gợi ý tối thiểu: trước khi viết, hãy ghi bất biến bằng một câu tiếng Việt; hàm bắt buộc trả đúng loại dữ liệu yêu cầu. Sau khi tự viết, bỏ `pass` và test thủ công với ví dụ và ca biên ở mục 10.


In [36]:
def exercise_number_171(columnTitle):
    # TODO 1: total=0
    # TODO 2: với từng ch, ánh xạ sang digit 1..26
    # TODO 3: total = total*26 + digit
    # TODO 4: return total
    pass


### Bài 4 — TODO biến thể

Viết `exercise_encode_171(n)` nhận số nguyên dương, trả tên cột (chiều ngược). Bắt buộc kiểm thử 26→Z, 27→AA, 52→AZ, 701→ZY.

Bạn được thay kiểu đầu ra nhưng không được thay định nghĩa bài gốc. Để hàm có chữ ký riêng và **không gọi tự động**; tạo dữ liệu thử sau khi đã có code.


In [37]:
def exercise_encode_171(n):
    # TODO: divmod(n-1,26), lấy chữ cuối và đảo kết quả
    pass


### Bài 5 — TODO tự tạo test có chủ đích

Tự nghĩ ít nhất 5 case: A, Z, AB, ZZ, AAA, FXSHRXW; thêm một ca khiến chương trình “cộng thay vì nhân” sai. Cả `expected` đều là số, không phải string.

Thử đặt câu hỏi: một chương trình sai phổ biến có thể vẫn vượt qua ca nào? Cần thêm ca nào để khiến nó lộ sai? Viết kiểm thử `assert` trong hàm độc lập, sau đó chỉ gọi khi đã hoàn thiện hàm TODO 3, 4.


In [38]:
def exercise_tests_171():
    # TODO: >=5 assert cho exercise_number_171
    pass


### Bài 6 — mini challenge chuyển giao kỹ thuật

Tổng quát hàm giải mã cho `alphabet` tùy ý gồm k≥2 ký hiệu **không lặp**: mỗi ký hiệu đại diện 1..k, công thức `total=total*k+digit`. Ví dụ alphabet ABC, BA=7.

Đây là bài nâng cao: bạn không cần giải nhanh. Hãy trình bày ý tưởng bằng tay, viết code sạch, tự mô phỏng input nhỏ rồi mới so sánh với đáp án tham khảo cuối notebook.


In [39]:
def exercise_decode_custom_171(s,alphabet):
    # TODO: duyệt chữ, tìm index trong alphabet +1
    pass


## 16. Gợi ý ba mức — chỉ mở sau khi tự thử

**Bài 1:** M1: A index Unicode. M2: +1 ánh xạ. M3: 1,26,1.

**Bài 2:** M1: nhân 26 khi thêm chữ. M2: AZ=1*26+26. M3: AZ=52, BA=53, AAA=703.

**Bài 3:** M1: `total=0`. M2: digit=ord(ch)-ord(A)+1. M3: `total=total*26+digit`.

**Bài 4:** M1: đây là #168. M2: số cột bắt đầu từ 1. M3: `n,r=divmod(n-1,26)`.

**Bài 5:** M1: nhiều ký tự Z. M2: quên nhân lộ sai ở AB. M3: case FXSHRXW ra 2147483647.

**Bài 6:** M1: cơ số k. M2: tìm digit qua alphabet. M3: dùng `alphabet.index(ch)+1`.

**Cách tự đánh giá:** Nếu mức 1 đã đủ để bạn làm được, hãy che mức 2–3. Nếu phải dùng mức 3, thử giải lại từ đầu sau khi đóng notebook. Học để có thể **tự suy ra**, không phải chép một đoạn code đã biết.


## 17. Đáp án tham khảo — đặt cuối để tránh lộ trước khi làm TODO

Phần này dùng **tên hàm `answer_*` khác với `exercise_*`** để notebook vẫn chạy an toàn. Đáp án không cần giống hệt từng ký tự code của bạn, nhưng phải giữ đúng điều kiện và chứng minh cùng độ phức tạp.

Đáp án bài 3 không dùng mũ vì tổng tiền tố đủ thông tin. Bài 4 giải ngược bằng phép chia đặc biệt. Bài 6 thay cơ số 26 bằng độ dài bảng ký hiệu.


In [40]:
def answer_number_171(s):
    total=0
    for ch in s:
        total=26*total + ord(ch)-ord('A')+1
    return total
for s,expected in [('AB',28),('ZY',701),('FXSHRXW',2147483647)]:
    assert answer_number_171(s)==expected
print('Đáp án bài 3: PASS')


Đáp án bài 3: PASS


### Đáp án bài 4 — suy ra từ biến thể

Đổi số về tên cột lấy từ phải: `n-1`, chia cho 26, phần dư r biểu diễn A..Z theo 0-based, rồi đảo chuỗi. Nếu bỏ trừ 1, các bội số 26 sẽ xuất hiện phần dư 0 không hợp lệ với Excel.

Hãy so sánh với phần bạn tự viết: code nào dễ nhìn ra bất biến hơn? Cả hai đúng vẫn có thể khác nhau về chi phí bộ nhớ hoặc sự rõ ràng.


In [41]:
def answer_encode_171(n):
    parts=[]
    while n>0:
        n,r=divmod(n-1,26)
        parts.append(chr(65+r))
    return ''.join(reversed(parts))
for s in ['A','AB','AZ','ZY','FXSHRXW']:
    assert answer_encode_171(answer_number_171(s))==s
print('Đáp án bài 4: PASS')


Đáp án bài 4: PASS


### Đáp án bài 6, và cách kiểm tra bài 5

Với bảng ký hiệu kích thước k≥2, mỗi chữ d từ 1..k. Mọi phần còn lại giữ nguyên phép gộp `total*k+d`. Khi dùng `index`, thời gian tra chữ có thể O(k) nếu alphabet lớn; nếu k là hằng nhỏ thì mỗi vòng coi O(1).

Mỗi test có một mục đích. Khi tự thiết kế test, ghi chú ca nào kiểm điều kiện dừng, ca nào kiểm cập nhật, ca nào kiểm dữ liệu hợp lệ khác thường.


In [42]:
def answer_custom_171(s,alphabet):
    assert len(alphabet)>=2 and len(set(alphabet))==len(alphabet)
    total=0
    for ch in s:
        total=total*len(alphabet)+alphabet.index(ch)+1
    return total
assert answer_custom_171('BA','ABC')==7
assert answer_custom_171('AA','ABC')==4
assert answer_custom_171('ZY','ABCDEFGHIJKLMNOPQRSTUVWXYZ')==701
print('Đáp án bài 6: PASS')


Đáp án bài 6: PASS


### Đáp án / lời giải thích bằng lời cho cả 6 bài

**Bài 1:** A →1, Z→26, giá trị `0*26+1=1`.

**Bài 2:** AZ=1*26+26=52; BA=2*26+1=53; AAA=(1*26+1)*26+1=703.

**Bài 3:** Chọn trạng thái tối thiểu như `solve_*`; đọc lại cell đáp án ngay trên và giải thích từng phép gán.

**Bài 4:** Đổi số→tên bằng `divmod(n-1,26)`, ánh xạ dư 0..25, rồi đảo chữ.

**Bài 5:** Dùng A, Z, AB, AZ, ZZ, AAA, FXSHRXW để bắt lỗi 0-based/nhân sai/giới hạn.

**Bài 6:** Thay 26 bằng k=len(alphabet), ánh xạ 1..k và giữ invariant tổng tiền tố.

> Nếu chưa thể tự diễn giải được các dòng `return`, hãy quay lại bảng trace và lặp lại bằng một input tự chọn.


## 18. Tổng kết — template suy nghĩ + checklist nộp bài

### Template nhớ nhanh cho #171

1. **Định nghĩa:** Nhận chuỗi tên cột uppercase A..Z, trả số thứ tự cột Excel.
2. **Trạng thái:** Tổng total là số cột của phần tiền tố đã đọc.
3. **Quy tắc cập nhật:** Đọc ch, digit=ord(ch)-ord(A)+1; total=26*total+digit.
4. **Bất biến:** Sau k chữ, total đúng bằng giá trị của k chữ đầu.
5. **Điều kiện dừng:** Sau khi duyệt hết ký tự, trả total.
6. **Độ phức tạp:** Time O(k), Extra Space O(1), k là độ dài chuỗi.

### Checklist tự đánh giá
- [ ] Tôi phát biểu được đề, output và ràng buộc mà không nhìn code.
- [ ] Tôi hiểu cú pháp mới và tự chạy được ít nhất hai thí nghiệm nhỏ.
- [ ] Tôi trình bày được baseline, điểm yếu và lý do cải tiến.
- [ ] Tôi tự trace được 4 bước, kể cả một nhánh đặc biệt.
- [ ] Tôi phân biệt được chứng minh thuật toán với kiểm thử.
- [ ] Tôi biết khai báo `class Solution` đúng tên hàm, đúng kiểu trả về.
- [ ] Tôi đã làm 6 bài tự luyện **trước khi** xem đáp án.
- [ ] Tôi kiểm tra lại edge cases, time/space, không để print debug khi submit.

**Câu hỏi cuối:** Nếu thay input và biến thể, tôi có tự viết lại thuật toán mà không nhìn notebook được không? Nếu có, bạn đã hiểu kỹ thuật, không chỉ thuộc code.
